# Figure S5

In [6]:
from riskhistory.format import rcParams
from riskhistory.data import (
    load_splithalf_choicehistory_betas,
    load_splithalf_ddm_params,
    load_splithalf_rtswitch_betas,
    DATASET_LABELS,
)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import pingouin as pg
import statsmodels.api as sm

DATASETS = ["equalIndifference", "equalRange", "sheng2020"]
HALVES = [1, 2]

### Helper functions

In [ ]:
BF_THRESHOLD_HIGH = 3
BF_THRESHOLD_LOW = 1 / 3

BF_COLORS = {
    "evidence_for":     "#017D01",
    "inconclusive":     "#B0B0B0",
    "evidence_against": "#000000",
}


def bf_category(bf10):
    if bf10 > BF_THRESHOLD_HIGH:
        return "evidence_for"
    if bf10 < BF_THRESHOLD_LOW:
        return "evidence_against"
    return "inconclusive"


def spearman_corr_bf(x, y):
    """Spearman rho, p and BF10 (Pearson on ranks), plus an OLS line and 95% CI on the raw points."""
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)

    res = pg.corr(pd.Series(x).rank().values, pd.Series(y).rank().values).iloc[0]
    bf10 = float(res["BF10"])

    ols_fit = sm.OLS(y, sm.add_constant(x)).fit()
    x_line = np.linspace(x.min(), x.max(), 100)
    pred = ols_fit.get_prediction(sm.add_constant(x_line)).summary_frame(alpha=0.05)

    return {
        "r": float(res["r"]),
        "pval": float(res["p_val"]),
        "bf10": bf10,
        "category": bf_category(bf10),
        "n": len(x),
        "x_line": x_line,
        "y_line": pred["mean"].values,
        "ci_low": pred["mean_ci_lower"].values,
        "ci_high": pred["mean_ci_upper"].values,
    }


def style_axis(ax, col=None, ylabel=None, xlabel=None, xlabel_col=1):
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    ax.set_ylabel(ylabel)
    if col == xlabel_col and xlabel:
        ax.set_xlabel(xlabel)
    else:
        ax.set_xlabel(" ")


def format_log_bf(bf10):
    return f"{np.log(bf10):.1f}"


def format_pval(p):
    if p < 0.001:
        return "p < .001"
    if p < 0.01:
        return "p < .01"
    if p < 0.05:
        return "p < .05"
    return "n.s."


def plot_spearman_corr(ax, x, y, ylim=None):
    result = spearman_corr_bf(x, y)
    color = BF_COLORS[result["category"]]

    ax.axhline(0, color="black", linestyle="--", linewidth=1, zorder=0)
    ax.scatter(x, y, color=color, alpha=0.75, edgecolor="black",
               linewidth=0.5, s=75, zorder=3)
    ax.plot(result["x_line"], result["y_line"], color="black", linewidth=1.5,
            linestyle="-", zorder=4)

    ax.text(0.05, 0.95,
            f"ρ = {result['r']:.2f}, {format_pval(result['pval'])}\n"
            f"ln(BF10) = {format_log_bf(result['bf10'])}\nn = {result['n']}",
            transform=ax.transAxes, ha="left", va="top", fontsize=16,
            color="black", fontweight="bold")

    if ylim:
        ax.set_ylim(ylim)

    return result


def ddm_shifts(ddm_params, ds, half_id):
    params = (ddm_params.query("dataset == @ds & half_id == @half_id")
              .pivot(index="sub", columns="param_name", values="param_value"))
    return 2 * params[["v_choice_prev1", "z_choice_prev1"]]


def behavior_effects(choicehistory_betas, switch_betas, ds, half_id):
    choice_effect = (choicehistory_betas.query("dataset == @ds & half_id == @half_id")
                     .set_index("sub")["choicehistory"].rename("choice_prev1_effect"))
    switch_effect = (switch_betas.query("dataset == @ds & half_id == @half_id")
                     .set_index("sub")["switch_effect"])
    return pd.concat([choice_effect, switch_effect], axis=1)


def crosshalf_data(ddm_params, choicehistory_betas, switch_betas, ds, ddm_half):
    behavior_half = 3 - ddm_half
    return pd.concat([ddm_shifts(ddm_params, ds, ddm_half),
                      behavior_effects(choicehistory_betas, switch_betas, ds, behavior_half)],
                     axis=1, join="inner")

### Figure: DDM shifts from one half vs. behavioral choice-history effects from the other half
Per-participant SP-Drift-CH pyddm fits (`analysis_splithalf/2_fitddm_splithalf.py`) on one half of the data (Sheng2020 run 1 / run 2; NARPS runs 1 and 4 / runs 2 and 3), against the prospect-theory repetition bias and the RT switch cost estimated on the other half. One figure per DDM half. Shifts are 2 x the ±1-coded `choice_prev1` effect (accept − reject), as in `figure3.ipynb`. Spearman ρ, with color by BF10 (green > 3, black < 1/3, gray otherwise); black line = OLS fit. Participants whose prospect-theory or RT fit did not converge are dropped.

In [ ]:
ddm_params = load_splithalf_ddm_params()
choicehistory_betas = load_splithalf_choicehistory_betas()
choicehistory_betas = choicehistory_betas[choicehistory_betas["success"]]
switch_betas = load_splithalf_rtswitch_betas()
switch_betas = switch_betas[switch_betas["converged"]]

for ddm_half in HALVES:
    fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 11), layout="constrained")

    for col, ds in enumerate(DATASETS):
        merged = crosshalf_data(ddm_params, choicehistory_betas, switch_betas, ds, ddm_half)
        axes[0, col].set_title(DATASET_LABELS.get(ds, ds))

        df = merged[["v_choice_prev1", "choice_prev1_effect"]].dropna()
        plot_spearman_corr(axes[0, col], df["v_choice_prev1"], df["choice_prev1_effect"], ylim=(-2, 2))
        style_axis(axes[0, col], col,
                   ylabel=f"Previous choice repetition bias\n(half {3 - ddm_half})",
                   xlabel=f"Choice history shift of the fixed-drift bias (∆v, half {ddm_half})")

        df = merged[["z_choice_prev1", "switch_effect"]].dropna()
        plot_spearman_corr(axes[1, col], df["z_choice_prev1"], df["switch_effect"], ylim=(-0.5, 0.5))
        style_axis(axes[1, col], col,
                   ylabel=f"Cost of choice switching (RT)\n(half {3 - ddm_half})",
                   xlabel=f"Choice history shift of the starting point bias (∆z, half {ddm_half})")

    for row, label in enumerate(["A.", "B."]):
        axes[row, 0].text(-0.3, 1.1, label, transform=axes[row, 0].transAxes,
                          fontsize=26, fontweight="bold", va="top", ha="right")

    fig.savefig(f"saved/figureS5_ddmhalf{ddm_half}.svg", dpi=300, bbox_inches="tight")
    plt.show()